<a href="https://colab.research.google.com/github/cmoney113/Mac_file_and_web_tool/blob/main/Mac_App_Mover_Enhancements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#!/usr/bin/env python3
import os
import sys
import shutil
import time
import datetime
import json
import subprocess
import tempfile
import platform
import re
from concurrent.futures import ThreadPoolExecutor

from PyQt5.QtWidgets import (
    QApplication,
    QMainWindow,
    QPushButton,
    QVBoxLayout,
    QHBoxLayout,
    QWidget,
    QLabel,
    QCheckBox,
    QFileDialog,
    QTreeWidget,
    QTreeWidgetItem,
    QMessageBox,
    QProgressBar,
    QLineEdit,
    QHeaderView,
    QSplitter,
    QFrame,
    QTabWidget,
    QStyle,
    QStyleFactory,
    QMenu,
    QAction,
    QSystemTrayIcon,
    QDialog,
    QRadioButton,
    QComboBox,
    QSlider,
    QToolBar,
    QStatusBar,
    QTextEdit,
    QGroupBox,
    QFormLayout,
    QSpinBox,
)
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QSize, QTimer, QSettings, QDateTime, QSortFilterProxyModel
from PyQt5.QtGui import QPalette, QColor, QFont, QIcon, QPixmap, QImage, QPainter, QBrush, QFontMetrics


class AppInfo:
    """Class to store app information"""

    def __init__(self, path):
        self.path = path
        self.name = os.path.basename(path)
        self.location = os.path.dirname(path)
        self.size = self.get_size()
        self.modified = self.get_modified_time()
        self.is_symlink = os.path.islink(path)
        self.target = os.readlink(path) if self.is_symlink else ""
        self.app_type = "Setapp" if "/Setapp/" in path else "Standard"
        self.version = self.get_version()
        self.last_used = self.get_last_used()
        self.icon = self.get_icon()  # Placeholder
        self.id = self.get_bundle_id()

    def get_size(self):
        """Calculate the size of the app bundle"""
        total_size = 0
        try:
            for dirpath, dirnames, filenames in os.walk(self.path):
                for f in filenames:
                    fp = os.path.join(dirpath, f)
                    if not os.path.islink(fp):
                        total_size += os.path.getsize(fp)
        except Exception:
            pass
        return total_size

    def get_modified_time(self):
        """Get the modified time of the app bundle"""
        try:
            return os.path.getmtime(self.path)
        except Exception:
            return 0

    def get_version(self):
        """Get app version using macOS specific APIs"""
        try:
            plist_path = os.path.join(self.path, "Contents", "Info.plist")
            if os.path.exists(plist_path):
                cmd = ["defaults", "read", plist_path, "CFBundleShortVersionString"]
                result = subprocess.run(cmd, capture_output=True, text=True)
                if result.returncode == 0:
                    return result.stdout.strip()
            return "Unknown"
        except Exception:
            return "Unknown"

    def get_last_used(self):
        """Get app last used date using macOS metadata"""
        try:
            cmd = ["mdls", "-name", "kMDItemLastUsedDate", self.path]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                match = re.search(r"kMDItemLastUsedDate\s*=\s*(.*)", result.stdout)
                if match:
                    return match.group(1).strip()
            return "Unknown"
        except Exception:
            return "Unknown"

    def get_icon(self):
        """Get app icon as QIcon"""
        # Placeholder:  Need macOS specific implementation.
        return None

    def get_bundle_id(self):
        """Get app bundle ID"""
        try:
            plist_path = os.path.join(self.path, "Contents", "Info.plist")
            if os.path.exists(plist_path):
                cmd = ["defaults", "read", plist_path, "CFBundleIdentifier"]
                result = subprocess.run(cmd, capture_output=True, text=True)
                if result.returncode == 0:
                    return result.stdout.strip()
            return "Unknown"
        except Exception:
            return "Unknown"

    def format_size(self):
        """Format size in human-readable format"""
        size = self.size
        for unit in ["B", "KB", "MB", "GB"]:
            if size < 1024.0:
                return f"{size:.2f} {unit}"
            size /= 1024.0
        return f"{size:.2f} TB"

    def format_modified(self):
        """Format modified time in human-readable format"""
        return datetime.datetime.fromtimestamp(self.modified).strftime('%Y-%m-%d %H:%M:%S')

    def to_dict(self):
        """Convert to dictionary for serialization"""
        return {
            "path": self.path,
            "name": self.name,
            "location": self.location,
            "size": self.size,
            "modified": self.modified,
            "is_symlink": self.is_symlink,
            "target": self.target,
            "app_type": self.app_type,
            "version": self.version,
            "id": self.id,
        }

    @classmethod
    def from_dict(cls, data):
        """Create AppInfo from dictionary"""
        app_info = cls(data["path"])
        app_info.name = data["name"]
        app_info.location = data["location"]
        app_info.size = data["size"]
        app_info.modified = data["modified"]
        app_info.is_symlink = data["is_symlink"]
        app_info.target = data["target"]
        app_info.app_type = data["app_type"]
        app_info.version = data.get("version", "Unknown")
        app_info.id = data.get("id", "Unknown")
        return app_info


class BackupManager:
    """Class to handle backup and restore operations"""

    def __init__(self, backup_dir=None):
        if backup_dir is None:
            # Default backup directory in user's home
            self.backup_dir = os.path.expanduser("~/Library/Application Support/Mac App Mover/Backups")
        else:
            self.backup_dir = backup_dir

        # Create backup directory if it doesn't exist
        os.makedirs(self.backup_dir, exist_ok=True)

    def create_backup(self, app_info_list):
        """Create a backup of the current app locations"""
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_file = os.path.join(self.backup_dir, f"backup_{timestamp}.json")

        # Create a list of serializable app infos
        backup_data = {
            "timestamp": timestamp,
            "apps": [app.to_dict() for app in app_info_list],
        }

        # Write to file
        with open(backup_file, "w") as f:
            json.dump(backup_data, f, indent=2)

        return backup_file

    def list_backups(self):
        """List all available backups"""
        backups = []
        for f in os.listdir(self.backup_dir):
            if f.startswith("backup_") and f.endswith(".json"):
                backup_path = os.path.join(self.backup_dir, f)
                try:
                    with open(backup_path, "r") as file:
                        data = json.load(file)
                        backups.append(
                            {
                                "file": f,
                                "path": backup_path,
                                "timestamp": data.get("timestamp", ""),
                                "app_count": len(data.get("apps", [])),
                            }
                        )
                except Exception:
                    # Skip invalid backup files
                    pass

        # Sort by timestamp (most recent first)
        backups.sort(key=lambda x: x["timestamp"], reverse=True)
        return backups

    def load_backup(self, backup_path):
        """Load a backup file and return the app info list"""
        with open(backup_path, "r") as f:
            data = json.load(f)

        # Convert dictionaries back to AppInfo objects
        app_infos = [AppInfo.from_dict(app_data) for app_data.get("apps", [])]
        return app_infos

    def restore_backup(self, backup_path):
        """Restore apps from a backup (simplified for this context)"""
        app_infos = self.load_backup(backup_path)
        results = []

        for app_info in app_infos:
            try:
                # Basic restore logic (more robust error handling needed in a real app)
                if app_info.is_symlink:
                    if not os.path.exists(app_info.path) and os.path.exists(app_info.target):
                        os.symlink(app_info.target, app_info.path)
                        results.append(f"Restored symlink: {app_info.name}")
                else:
                    if os.path.exists(app_info.target) and not os.path.exists(app_info.path):
                         shutil.move(app_info.target, app_info.path)
                         results.append(f"Restored app: {app_info.name}")
                    elif os.path.exists(app_info.target) and os.path.islink(app_info.path):
                        os.unlink(app_info.path)
                        shutil.move(app_info.target, app_info.path)
                        results.append(f"Restored app: {app_info.name}")
                    else:
                        results.append(f"Skipped: {app_info.name} -  path exists")

            except Exception as e:
                results.append(f"Error restoring {app_info.name}: {str(e)}")
        return results


class LogManager:
    """Class to handle logging of operations"""

    def __init__(self, log_dir=None):
        if log_dir is None:
            # Default log directory
            self.log_dir = os.path.expanduser("~/Library/Application Support/Mac App Mover/Logs")
        else:
            self.log_dir = log_dir

        # Create log directory if it doesn't exist
        os.makedirs(self.log_dir, exist_ok=True)

        # Current log file
        self.current_log_file = os.path.join(self.log_dir, f"log_{datetime.datetime.now().strftime('%Y%m%d')}.log")
        self.executor = ThreadPoolExecutor(max_workers=1)  # Use a thread pool

    def log(self, message, level="INFO"):
        """Log a message"""
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{timestamp}] [{level}] {message}\n"
        try:
             self.executor.submit(self._write_log_entry, log_entry) # Use thread pool.
        except Exception as e:
            print(f"Error submitting log task: {e}")  # Log to console if thread submission fails

    def _write_log_entry(self, log_entry):
        """Helper to write to log file"""
        try:
            with open(self.current_log_file, "a") as f:
                f.write(log_entry)
        except Exception as e:
            print(f"Error writing to log file: {e}")

    def log_operation(self, operation, app_info, status, details=""):
        """Log an operation on an app"""
        message = f"{operation}: {app_info.name} - {status}"
        if details:
            message += f" - {details}"
        self.log(message)

    def get_recent_logs(self, count=100):
        """Get the most recent log entries"""
        if not os.path.exists(self.current_log_file):
            return []

        try:
            with open(self.current_log_file, "r") as f:
                lines = f.readlines()
            return lines[-count:]
        except Exception as e:
            print(f"Error reading log file: {e}")
            return []

    def list_log_files(self):
        """List all available log files"""
        log_files = []
        for f in os.listdir(self.log_dir):
            if f.startswith("log_") and f.endswith(".log"):
                log_path = os.path.join(self.log_dir, f)
                log_files.append(
                    {
                        "file": f,
                        "path": log_path,
                        "date": f[4:12],  # Extract date from filename
                        "size": os.path.getsize(log_path),
                    }
                )
        # Sort by date (most recent first)
        log_files.sort(key=lambda x["date"], reverse=True)
        return log_files

    def clear_logs(self):
        """Clear all log files"""
        try:
            for log_file in self.list_log_files():
                os.remove(log_file['path'])
            self.log("All logs cleared", level="INFO")
            return True
        except Exception as e:
            self.log(f"Error clearing logs: {e}", level="ERROR")
            return False


class DiskSpaceAnalyzer:
    """Class to analyze disk space"""

    @staticmethod
    def get_disk_info(path):
        """Get disk info for the given path"""
        try:
            stat = os.statvfs(path)
            free_bytes = stat.f_frsize * stat.f_bavail
            total_bytes = stat.f_frsize * stat.f_blocks
            used_bytes = total_bytes - free_bytes

            return {
                "path": path,
                "free_bytes": free_bytes,
                "free_formatted": DiskSpaceAnalyzer.format_size(free_bytes),
                "total_bytes": total_bytes,
                "total_formatted": DiskSpaceAnalyzer.format_size(total_bytes),
                "used_bytes": used_bytes,
                "used_formatted": DiskSpaceAnalyzer.format_size(used_bytes),
                "percent_used": (used_bytes / total_bytes) * 100 if total_bytes > 0 else 0,
            }
        except Exception:
            return {
                "path": path,
                "error": "Could not determine disk space",
            }

    @staticmethod
    def format_size(size_bytes):
        """Format size in human-readable format"""
        for unit in ["B", "KB", "MB", "GB", "TB"]:
            if size_bytes < 1024.0:
                return f"{size_bytes:.2f} {unit}"
            size_bytes /= 1024.0
        return f"{size_bytes:.2f} PB"

    @staticmethod
    def calculate_move_space_impact(selected_apps, source_path, target_path):
        """Calculate space impact of moving selected apps"""
        source_disk = DiskSpaceAnalyzer.get_disk_info(source_path)
        target_disk = DiskSpaceAnalyzer.get_disk_info(target_path)

        # Check if source and target are on the same disk
        same_disk = os.stat(source_path).st_dev == os.stat(target_path).st_dev

        # Calculate total size of selected apps
        total_size = sum(app.size for app in selected_apps)

        result = {
            "source_disk": source_disk,
            "target_disk": target_disk,
            "total_app_size": total_size,
            "total_app_size_formatted": DiskSpaceAnalyzer.format_size(total_size),
            "same_disk": same_disk,
        }

        if not same_disk:
            # Calculate impact
            result["source_impact"] = {
                "freed_bytes": total_size,
                "freed_formatted": DiskSpaceAnalyzer.format_size(total_size),
                "new_free_bytes": source_disk.get("free_bytes", 0) + total_size,
                "new_free_formatted": DiskSpaceAnalyzer.format_size(source_disk.get("free_bytes", 0) + total_size),
                "new_percent_used": (
                    (source_disk.get("used_bytes", 0) - total_size) / source_disk.get("total_bytes", 1)
                )
                * 100,
            }

            result["target_impact"] = {
                "used_bytes": total_size,
                "used_formatted": DiskSpaceAnalyzer.format_size(total_size),
                "new_free_bytes": target_disk.get("free_bytes", 0) - total_size,
                "new_free_formatted": DiskSpaceAnalyzer.format_size(
                    target_disk.get("free_bytes", 0) - total_size
                ),
                "new_percent_used": (
                    (target_disk.get("used_bytes", 0) + total_size) / target_disk.get("total_bytes", 1)
                )
                * 100,
            }

        return result



class ScanThread(QThread):
    """Thread for scanning directories to avoid freezing the UI"""

    scan_update = pyqtSignal(str)
    scan_app_found = pyqtSignal(object)
    scan_progress = pyqtSignal(int)
    scan_finished = pyqtSignal()

    def __init__(self, scan_dirs):
        super().__init__()
        self.scan_dirs = scan_dirs
        self.stopped = False
        self.executor = ThreadPoolExecutor(max_workers=4)  # Thread pool for scanning

    def stop(self):
        """Stop the scanning process"""
        self.stopped = True

    def run(self):
        total_dirs = len(self.scan_dirs)
        futures = []
        for i, directory in enumerate(self.scan_dirs):
            if self.stopped:
                break
            # Use a thread from the pool to scan each directory
            future = self.executor.submit(self.scan_directory, directory)
            futures.append(future)

            # Update progress
            progress = int(((i + 1) / total_dirs) * 100)
            self.scan_progress.emit(progress)

        # Wait for all scanning tasks to complete
        for future in futures:
            future.result()  #  retrieve the result (or exception)

        self.executor.shutdown(wait=True)
        self.scan_finished.emit()

    def scan_directory(self, directory):
        try:
            self.scan_update.emit(f"Scanning {directory}...")
            for root, dirs, files in os.walk(directory):
                if self.stopped:
                    break

                for name in dirs:
                    if self.stopped:
                        break
                    if name.endswith(".app"):
                        full_path = os.path.join(root, name)
                        self.scan_update.emit(f"Found: {full_path}")
                        # Create AppInfo object and emit it
                        app_info = AppInfo(full_path)
                        self.scan_app_found.emit(app_info)
        except Exception as e:
            self.scan_update.emit(f"Error scanning {directory}: {str(e)}")



class MoveThread(QThread):
    """Thread for moving apps to avoid freezing the UI"""

    move_update = pyqtSignal(str)
    move_progress = pyqtSignal(int)
    move_app_updated = pyqtSignal(object)  # Updated app info
    move_log = pyqtSignal(str, str, object, str)  # operation, status, app_info, details
    move_finished = pyqtSignal(dict)  # Summary of operations

    def __init__(self, apps_to_move, target_dirs, create_backup=True, logger=None):
        super().__init__()
        self.apps_to_move = apps_to_move  # List of AppInfo objects
        self.target_dirs = target_dirs
        self.create_backup = create_backup
        self.logger = logger
        self.stopped = False
        self.summary = {
            "total": len(apps_to_move),
            "successful": 0,
            "failed": 0,
            "skipped": 0,
            "total_size_moved": 0,
        }
        self.executor = ThreadPoolExecutor(max_workers=4)  # Thread pool for moving

    def stop(self):
        """Stop the moving process"""
        self.stopped = True
        self.executor.shutdown(wait=False)  #  don't wait for completion

    def run(self):
        # Create backup if needed
        if self.create_backup and self.logger:
            backup_manager = BackupManager()
            backup_file = backup_manager.create_backup(self.apps_to_move)
            self.move_update.emit(f"Created backup: {backup_file}")
            self.move_log.emit("BACKUP", "Created", None, backup_file)

        total = len(self.apps_to_move)
        futures = []
        for i, app_info in enumerate(self.apps_to_move):
            if self.stopped:
                self.move_update.emit("Operation stopped by user")
                break

            # Use a thread from the pool to move each app
            future = self.executor.submit(self.move_app, app_info)
            futures.append(future)

            # Update progress
            if not self.stopped:
                progress = int(((i + 1) / total) * 100)
                self.move_progress.emit(progress)

        # Wait for all move tasks to complete
        for future in futures:
            future.result()

        self.executor.shutdown(wait=True)
        self.move_finished.emit(self.summary)

    def move_app(self, app_info):
        """Move a single app and create a symlink.  This is done in a thread."""
        try:
            # Skip if it's already a symlink
            if app_info.is_symlink:
                self.move_update.emit(f"Skipping {app_info.name} as it's already a symlink")
                self.move_log.emit("MOVE", "SKIPPED", app_info, "Already a symlink")
                self.summary["skipped"] += 1
                return

            # Get target directory
            target_dir = self.get_target_directory(app_info)

            app_path = app_info.path
            app_name = app_info.name
            target_path = os.path.join(target_dir, app_name)

            # Check if target already exists
            if os.path.exists(target_path):
                self.move_update.emit(f"Error: Target already exists: {target_path}")
                self.move_log.emit("MOVE", "ERROR", app_info, f"Target already exists: {target_path}")
                self.summary["failed"] += 1
                return

            # Create target directory if it doesn't exist
            os.makedirs(target_dir, exist_ok=True)

            # 1. Move the app to the target directory
            self.move_update.emit(f"Moving {app_name} to {target_dir}")
            app_size = app_info.size
            shutil.move(app_path, target_path)

            # 2. Create symlink in original location
            self.move_update.emit(f"Creating symlink for {app_name}")
            os.symlink(target_path, app_path)

            # 3. Update app_info object with new information
            updated_app_info = AppInfo(app_path)  # This will now be a symlink
            self.move_app_updated.emit(updated_app_info)

            self.move_update.emit(f"Successfully moved and linked {app_name}")
            self.move_log.emit("MOVE", "SUCCESS", app_info, f"Moved to {target_path}")

            self.summary["successful"] += 1
            self.summary["total_size_moved"] += app_size
        except Exception as e:
            self.move_update.emit(f"Error processing {app_info.name}: {str(e)}")
            self.move_log.emit("MOVE", "ERROR", app_info, str(e))
            self.summary["failed"] += 1

    def get_target_directory(self, app_info):
        """Determine the target directory for a given app."""
        if app_info.app_type == "Setapp":
            return self.target_dirs["setapp"]
        else:
            return self.target_dirs["applications"]



class RestoreThread(QThread):
    """Thread for restoring apps from backup"""

    restore_update = pyqtSignal(str)
    restore_progress = pyqtSignal(int)
    restore_log = pyqtSignal(str, str, str)  # app_name, status, details
    restore_finished = pyqtSignal(dict)  # Summary of operations

    def __init__(self, backup_path, logger=None):
        super().__init__()
        self.backup_path = backup_path
        self.logger = logger
        self.stopped = False
        self.executor = ThreadPoolExecutor(max_workers=4)

    def stop(self):
        """Stop the restore process"""
        self.stopped = True
        self.executor.shutdown(wait=False)

    def run(self):
        backup_manager = BackupManager()
        self.restore_update.emit(f"Loading backup: {self.backup_path}")

        try:
            app_infos = backup_manager.load_backup(self.backup_path)
            self.restore_update.emit(f"Found {len(app_infos)} apps in backup")

            total = len(app_infos)
            futures = []
            for i, app_info in enumerate(app_infos):
                if self.stopped:
                    self.restore_update.emit("Operation stopped by user")
                    break
                future = self.executor.submit(self.restore_app, app_info)
                futures.append(future)

                # Update progress
                progress = int(((i + 1) / total) * 100)
                self.restore_progress.emit(progress)

            # Wait for all restore tasks to complete
            for future in futures:
                future.result()

            self.executor.shutdown(wait=True)
            self.restore_finished.emit(self.summary)

        except Exception as e:
            self.restore_update.emit(f"Error loading backup: {str(e)}")
            self.restore_finished.emit({"error": str(e), "backup_path": self.backup_path})

    def restore_app(self, app_info):
        """Restores a single app"""
        try:
             # Check if current path exists
            current_path = app_info.path
            current_exists = os.path.exists(current_path)
            current_is_symlink = os.path.islink(current_path) if current_exists else False
            success = False

            if not current_exists:
                # Path doesn't exist - create if symlink in backup
                if app_info.is_symlink and os.path.exists(app_info.target):
                    os.symlink(app_info.target, current_path)
                    self.restore_update.emit(f"Created symlink for {app_info.name}")
                    self.restore_log.emit(app_info.name, "RESTORED", f"Created symlink to {app_info.target}")
                    success = True
                else:
                    self.restore_update.emit(f"Cannot restore {app_info.name}, original path doesn't exist")
                    self.restore_log.emit(app_info.name, "SKIPPED", "Original path doesn't exist")
            elif current_is_symlink != app_info.is_symlink:
                # Need to restore
                if current_is_symlink:
                    # It's a symlink now but wasn't in backup
                    target = os.readlink(current_path)
                    os.unlink(current_path)

                    # Check if the target of the symlink exists
                    if os.path.exists(target):
                        # Move target back to original path
                        shutil.move(target, current_path)
                        self.restore_update.emit(f"Restored {app_info.name} from symlink to original")
                        self.restore_log.emit(app_info.name, "RESTORED", "Converted from symlink to original")
                        success = True
                    else:
                        self.restore_update.emit(f"Cannot restore {app_info.name}, target doesn't exist")
                        self.restore_log.emit(app_info.name, "FAILED", "Target of symlink doesn't exist")
                else:
                    # It's a real app now but was a symlink in backup
                    if os.path.exists(app_info.target):
                        # Backup current app
                        temp_path = f"{current_path}.bak"
                        shutil.move(current_path, temp_path)

                        # Create symlink
                        os.symlink(app_info.target, current_path)
                        self.restore_update.emit(f"Restored {app_info.name} symlink, original app backed up to {temp_path}")
                        self.restore_log.emit(app_info.name, "RESTORED",
                                            f"Created symlink, original backed up to {temp_path}")
                        success = True
                    else:
                        self.restore_update.emit(f"Cannot restore {app_info.name} symlink, target doesn't exist")
                        self.restore_log.emit(app_info.name, "FAILED", "Symlink target doesn't exist")
            else:
                # No change needed
                self.restore_update.emit(f"No change needed for {app_info.name}")
                self.restore_log.emit(app_info.name, "SKIPPED", "No change needed")
                success= True

            if success:
                self.summary['successful'] +=1
            else:
                self.summary['failed'] += 1

        except Exception as e:
            self.restore_update.emit(f"Error restoring {app_info.name}: {str(e)}")
            self.restore_log.emit(app_info.name, "ERROR", str(e))
            self.summary['failed'] += 1



class ExportThread(QThread):
    """Thread for exporting app info to CSV/JSON"""

    export_update = pyqtSignal(str)
    export_progress = pyqtSignal(int)
    export_finished = pyqtSignal(str)  # Path to exported file

    def __init__(self, app_info_list, export_path, format="csv"):
        super().__init__()
        self.app_info_list = app_info_list
        self.export_path = export_path
        self.format = format.lower()

    def run(self):
        try:
            self.export_update.emit(f"Exporting {len(self.app_info_list)} apps to {self.format.upper()}")

            if self.format == "csv":
                self._export_csv()
            elif self.format == "json":
                self._export_json()
            else:
                raise ValueError(f"Unsupported export format: {self.format}")

            self.export_update.emit(f"Export complete: {self.export_path}")
            self.export_finished.emit(self.export_path)

        except Exception as e:
            self.export_update.emit(f"Error exporting: {str(e)}")
            self.export_finished.emit("")

    def _export_csv(self):
        """Export app info to CSV"""
        import csv

        with open(self.export_path, "w", newline="") as f:
            writer = csv.writer(f)

            # Write header
            writer.writerow(
                [
                    "Name",
                    "Type",
                    "Size (Bytes)",
                    "Size (Formatted)",
                    "Location",
                    "Modified Date",
                    "Is Symlink",
                    "Target",
                    "Version",
                    "Last Used",
                    "Bundle ID",
                ]
            )

            # Write data
            total = len(self.app_info_list)
            for i, app_info in enumerate(self.app_info_list):
                writer.writerow(
                    [
                        app_info.name,
                        app_info.app_type,
                        app_info.size,
                        app_info.format_size(),
                        app_info.location,
                        app_info.format_modified(),
                        "Yes" ifapp_info.is_symlink else "No",
                        app_info.target,
                        app_info.version,
                        app_info.last_used,
                        app_info.id,
                    ]
                )

                # Update progress
                progress = int(((i + 1) / total) * 100)
                self.export_progress.emit(progress)

    def _export_json(self):
        """Export app info to JSON"""
        data = {"exported_at": datetime.datetime.now().isoformat(), "apps": [app.to_dict() for app in self.app_info_list]}

        with open(self.export_path, "w") as f:
            json.dump(data, f, indent=2)

        self.export_progress.emit(100)



class AppMover(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Mac App Mover")
        self.setMinimumSize(1200, 800)

        # Default directories
        self.scan_dirs = ["/Applications", "/Applications/Setapp"]
        self.target_dirs = {
            "setapp": "/Volumes/SSDMAC/Apps/Setapp_SSD",
            "applications": "/Volumes/SSDMAC/Apps/ApplicationsSSD",
        }
        self.target_categories = {}  # Add this line

        # App data
        self.app_info_list = []
        self.filtered_app_list = []
        self.selected_category = "All"

        # Create managers
        self.backup_manager = BackupManager()
        self.log_manager = LogManager()

        # Set the dark navy theme by default
        self.is_dark_theme = True
        self.set_dark_navy_theme()

        # Load settings
        self.settings = QSettings("MacUtils", "Mac App Mover")
        self.load_settings()

        # Create system tray icon
        self.create_tray_icon()

        # Create UI
        self.init_ui()

        # Current threads
        self.scan_thread = None
        self.move_thread = None
        self.restore_thread = None
        self.export_thread = None

        # Log startup
        self.log_manager.log("Application started")

    def load_settings(self):
        """Load settings from QSettings"""
        # Load scan directories
        size = self.settings.beginReadArray("scan_dirs")
        if size > 0:
            self.scan_dirs = []
            for i in range(size):
                self.settings.setArrayIndex(i)
                dir_path = self.settings.value("path")
                if dir_path and os.path.exists(dir_path):
                    self.scan_dirs.append(dir_path)
        self.settings.endArray()

        # If no valid scan dirs, use defaults
        if not self.scan_dirs:
            self.scan_dirs = ["/Applications", "/Applications/Setapp"]

        # Load target directories
        self.target_dirs["setapp"] = self.settings.value("target_setapp", "/Volumes/SSDMAC/Apps/Setapp_SSD")
        self.target_dirs["applications"] = self.settings.value(
            "target_applications", "/Volumes/SSDMAC/Apps/ApplicationsSSD"
        )
        self.target_categories = self.settings.value("target_categories", {})  # Load

        # Convert loaded data to the correct dictionary structure if necessary
        if not isinstance(self.target_categories, dict):
            self.target_categories = {}

        # Ensure that each category has a 'path'
        for key, value in self.target_categories.items():
            if not isinstance(value, dict):
                self.target_categories[key] = {'path': value}

        # Load theme preference
        self.is_dark_theme = self.settings.value("dark_theme", True, type=bool)

        # Load window state/geometry if available
        if self.settings.contains("windowGeometry"):
            self.restoreGeometry(self.settings.value("windowGeometry"))
        if self.settings.contains("windowState"):
            self.restoreState(self.settings.value("windowState"))

    def save_settings(self):
        """Save settings to QSettings"""
        # Save scan directories
        self.settings.beginWriteArray("scan_dirs")
        for i, dir_path in enumerate(self.scan_dirs):
            self.settings.setArrayIndex(i)
            self.settings.setValue("path", dir_path)
        self.settings.endArray()

        # Save target directories
        self.settings.setValue("target_setapp", self.target_dirs["setapp"])
        self.settings.setValue("target_applications", self.target_dirs["applications"])
        self.settings.setValue("target_categories", self.target_categories)  # Save

        # Save theme preference
        self.settings.setValue("dark_theme", self.is_dark_theme)

        # Save window state/geometry
        self.settings.setValue("windowGeometry", self.saveGeometry())
        self.settings.setValue("windowState", self.saveState())

    def closeEvent(self, event):
        """Handle window close event"""
        self.save_settings()

        # Hide to tray if enabled
        if hasattr(self, "tray_icon") and self.tray_icon.isVisible():
            # Minimize to tray instead of closing
            event.ignore()
            self.hide()
            self.tray_icon.showMessage(
                "Mac App Mover", "Application minimized to system tray", QSystemTrayIcon.Information, 2000
            )
        else:
            # Normal close
            event.accept()
            # Stop any running threads
            self._stop_running_threads()

    def _stop_running_threads(self):
        """Stop any running threads"""
        if self.scan_thread and self.scan_thread.isRunning():
            self.scan_thread.stop()
            self.scan_thread.wait()

        if self.move_thread and self.move_thread.isRunning():
            self.move_thread.stop()
            self.move_thread.wait()

        if self.restore_thread and self.restore_thread.isRunning():
            self.restore_thread.stop()
            self.restore_thread.wait()

        if self.export_thread and self.export_thread.isRunning():
            self.export_thread.wait()  # Just wait for export to finish

    def create_tray_icon(self):
        """Create system tray icon"""
        self.tray_icon_menu = QMenu(self)

        # Show/hide action
        self.show_action = QAction("Show", self)
        self.show_action.triggered.connect(self.show)
        self.tray_icon_menu.addAction(self.show_action)

        # Exit action
        self.quit_action = QAction("Exit", self)
        self.quit_action.triggered.connect(self.real_quit)
        self.tray_icon_menu.addAction(self.quit_action)

        # Create tray icon
        self.tray_icon = QSystemTrayIcon(self)
        self.tray_icon.setContextMenu(self.tray_icon_menu)

        # Set icon
        icon = QIcon(":/appicon.png")  #  Replace with your icon path.
        if icon.isNull():
            icon = self.style().standardIcon(QStyle.SP_ComputerIcon)
        self.tray_icon.setIcon(icon)

        # Show the tray icon
        self.tray_icon.show()

        # Connect activated signal
        self.tray_icon.activated.connect(self.tray_icon_activated)

    def tray_icon_activated(self, reason):
        """Handle tray icon activation"""
        if reason == QSystemTrayIcon.DoubleClick:
            if self.isVisible():
                self.hide()
            else:
                self.show()
                self.activateWindow()

    def real_quit(self):
        """Really quit the application"""
        self.save_settings()
        self._stop_running_threads()
        QApplication.quit()

    def set_dark_navy_theme(self):
        """Set a dark navy theme that matches macOS style"""
        # Use Fusion style as base
        QApplication.setStyle(QStyleFactory.create("Fusion"))

        # Dark color palette with navy blue accents
        dark_palette = QPalette()

        # Base colors
        navy_dark = QColor(20, 35, 55)  # Background
        navy_mid = QColor(30, 50, 80)  # Mid tones
        navy_light = QColor(60, 80, 120)  # Highlight
        navy_accent = QColor(95, 125, 175)  # Accent color
        text_light = QColor(225, 230, 235)  # Light text
        text_dark = QColor(45, 60, 90)  # Dark text for light backgrounds

        # Set colors for different UI elements
        dark_palette.setColor(QPalette.Window, navy_dark)
        dark_palette.setColor(QPalette.WindowText, text_light)
        dark_palette.setColor(QPalette.Base, QColor(15, 25, 40))
        dark_palette.setColor(QPalette.AlternateBase, navy_mid)
        dark_palette.setColor(QPalette.ToolTipBase, navy_light)
        dark_palette.setColor(QPalette.ToolTipText, text_light)
        dark_palette.setColor(QPalette.Text, text_light)
        dark_palette.setColor(QPalette.Button, navy_mid)
        dark_palette.setColor(QPalette.ButtonText, text_light)
        dark_palette.setColor(QPalette.BrightText, Qt.white)
        dark_palette.setColor(QPalette.Link, navy_accent)
        dark_palette.setColor(QPalette.Highlight, navy_light)
        dark_palette.setColor(QPalette.HighlightedText, Qt.white)

        # Disabled colors
        dark_palette.setColor(QPalette.Disabled, QPalette.WindowText, QColor(100, 110, 130))
        dark_palette.setColor(QPalette.Disabled, QPalette.Text, QColor(100, 110, 130))
        dark_palette.setColor(QPalette.Disabled, QPalette.ButtonText, QColor(100, 110, 130))

        # Apply the palette
        QApplication.setPalette(dark_palette)

        # Set stylesheet for fine-tuning.  This is critical for more modern look.
        style_sheet = """
            QMainWindow, QDialog {
                background-color: #14233A;
            }
            QLabel {
                font-size: 13px;
                color: #E1E6EB;
            }
            QLabel#title_label {
                font-size: 18px;
                font-weight: bold;
                color: #FFFFFF;
                padding: 10px;
            }
            QLabel#section_label {
                font-size: 15px;
                font-weight: bold;
                color: #FFFFFF;
                padding: 5px 0;
            }
            QLineEdit, QComboBox, QSpinBox {
                background-color: #1A293D;
                border: 1px solid #3C5078;
                border-radius: 4px;
                padding: 6px;
                selection-background-color: #3A5078;
                font-size: 13px;
                color: #E1E6EB;
            }
            QLineEdit:focus, QComboBox:focus, QSpinBox:focus {
                border: 1px solid #5F7DAF;
                box-shadow: 0 0 5px rgba(95, 125, 175, 0.5);
            }
            QComboBox::drop-down {
                subcontrol-origin: padding;
                subcontrol-position: top right;
                width: 20px;
                border-left: 1px solid #3C5078;
            }
            QComboBox QAbstractItemView {
                border: 1px solid #3C5078;
                background-color: #14233A;
                selection-background-color: #3C5078;
                color: #E1E6EB;
                font-size: 13px;
            }
            QPushButton {
                background-color: #3C5078;
                color: #FFFFFF;
                border: none;
                border-radius: 6px;
                padding: 10px 20px;
                font-size: 13px;
                font-weight: bold;
                transition: background-color 0.3s ease;
            }
            QPushButton:hover {
                background-color: #4C6592;
            }
            QPushButton:pressed {
                background-color: #5F7DAF;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) inset;
            }
            QPushButton:disabled {
                background-color: #2A3A50;
                color: #647A9E;
            }
            QPushButton#primary_button {
                background-color: #4A7ACF;
            }
            QPushButton#primary_button:hover {
                background-color: #5A8ADF;
            }
            QPushButton#primary_button:pressed {
                background-color: #6A9AEF;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) inset;
            }
            QPushButton#danger_button {
                background-color: #CF4A4A;
                transition: background-color 0.3s ease;
            }
            QPushButton#danger_button:hover {
                background-color: #DF5A5A;
            }
            QPushButton#danger_button:pressed {
                background-color: #EF6A6A;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) inset;
            }

            QProgressBar {
                border: 1px solid #3C5078;
                border-radius: 6px;
                text-align: center;
                background-color: #1A293D;
                height: 24px;
            }
            QProgressBar::chunk {
                background-color: #5F7DAF;
                border-radius: 4px;
                transition: width 0.5s ease;
            }
            QTreeWidget, QTextEdit, QListWidget {
                background-color: #0F1928;
                alternate-background-color: #162438;
                border: 1px solid #3C5078;
                border-radius: 6px;
                selection-background-color: #3A5078;
                font-size: 13px;
                color: #E1E6EB;
            }
            QTreeWidget::item, QListWidget::item {
                padding: 8px;
            }
            QTreeWidget::item:selected, QListWidget::item:selected {
                background-color: #3C5078;
                border-radius: 4px;
            }
            QHeaderView::section {
                background-color: #1E304D;
                color: #FFFFFF;
                padding: 8px;
                border: none;
                border-right: 1px solid #3C5078;
                font-size: 13px;
                font-weight: bold;
            }
            QHeaderView::section:last {
                border-right: none;
            }
            QSplitter::handle {
                background-color: #3C5078;
                width: 1px;
            }
            QSplitter::handle:hover {
                background-color: #5F7DAF;
            }
            QTabWidget::pane {
                border: 1px solid #3C5078;
                border-radius: 6px;
                background-color: #14233A;
            }
            QTabBar::tab {
                background-color: #1E304D;
                color: #ADBACF;
                border-top-left-radius: 6px;
                border-top-right-radius: 6px;
                padding: 10px 20px;
                font-size: 13px;
                transition: background-color 0.3s ease, color 0.3s ease;
            }
            QTabBar::tab:selected {
                background-color: #3C5078;
                color: #FFFFFF;
            }
            QTabBar::tab:hover:!selected {
                background-color: #263A59;
                color: #FFFFFF;
            }
            QToolBar {
                background-color: #1E304D;
                border: none;
                spacing: 10px;
                padding: 10px;
                border-radius: 6px;
            }
            QToolBar QToolButton {
                background-color: transparent;
                border-radius: 6px;
                padding: 8px;
                color: #E1E6EB;
                transition: background-color 0.3s ease;
            }
            QToolBar QToolButton:hover {
                background-color: #3C5078;
                color: #FFFFFF;
            }
            QToolBar QToolButton:pressed {
                background-color: #5F7DAF;
                color: #FFFFFF;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) inset;
            }
            QToolBar QToolButton:checked {
                background-color: #3C5078;
                color: #FFFFFF;
            }
            QStatusBar {
                background-color: #1E304D;
                color: #E1E6EB;
                padding: 8px;
                border-top: 1px solid #3C5078;
            }
            QStatusBar::item {
                border: none;
            }
            QMenu {
                background-color: #14233A;
                border: 1px solid #3C5078;
                border-radius: 6px;
                padding: 5px;
                font-size: 13px;
                color: #E1E6EB;
            }
            QMenu::item {
                padding: 8px 25px 8px 30px;
                color: #E1E6EB;
                transition: background-color 0.3s ease, color 0.3s ease;
            }
            QMenu::item:selected {
                background-color: #3C5078;
                color: #FFFFFF;
                border-radius: 4px;
            }
            QMenu::separator {
                height: 1px;
                background-color: #3C5078;
                margin: 8px 0;
            }
            QSlider::groove:horizontal {
                height: 10px;
                background-color: #1A293D;
                border-radius: 5px;
                margin: 0;
            }
            QSlider::handle:horizontal {
                background-color: #5F7DAF;
                border: none;
                width: 20px;
                height: 20px;
                margin: -5px 0;
                border-radius: 10px;
                transition: transform 0.2s ease;
            }
            QSlider::handle:horizontal:hover {
                transform: scale(1.1);
            }
            QSlider::add-page:horizontal {
                background-color: #1A293D;
                border-radius: 5px;
            }
            QSlider::sub-page:horizontal {
                background-color: #3C5078;
                border-radius: 5px;
            }
            QCheckBox {
                color: #E1E6EB;
                font-size: 13px;
            }
            QCheckBox::indicator {
                width: 20px;
                height: 20px;
                background-color: #1A293D;
                border: 1px solid #3C5078;
                border-radius: 4px;
                transition: background-color 0.2s ease, border-color 0.2s ease;
            }
            QCheckBox::indicator:hover {
                border-color: #5F7DAF;
            }
            QCheckBox::indicator:checked {
                background-color: #5F7DAF;
                border-color: #5F7DAF;
                border-radius: 4px;
            }
            QCheckBox::indicator:checked:disabled {
                background-color: #647A9E;
                border-color: #647A9E;
            }
            QCheckBox::indicator:disabled {
                border-color: #3C5078;
            }
            QRadioButton {
                color: #E1E6EB;
                font-size: 13px;
            }
            QRadioButton::indicator {
                width: 20px;
                height: 20px;
                background-color: #1A293D;
                border: 1px solid #3C5078;
                border-radius: 10px;
                transition: background-color 0.2s ease, border-color 0.2s ease;
            }
            QRadioButton::indicator:hover {
                border-color: #5F7DAF;
            }
            QRadioButton::indicator:checked {
                background-color: #5F7DAF;
                border-color: #5F7DAF;
                border-radius: 10px;
            }
            QRadioButton::indicator:checked:disabled {
                background-color: #647A9E;
                border-color: #647A9E;
            }
            QRadioButton::indicator:disabled {
                border-color: #3C5078;
            }
            QGroupBox {
                border: 1px solid #3C5078;
                border-radius: 6px;
                margin-top: 24px;
                padding-top: 18px;
                font-size: 14px;
                font-weight: bold;
                color: #FFFFFF;
            }
            QGroupBox::title {
                subcontrol-origin: margin;
                subcontrol-position: top left;
                padding: 0 10px;
                margin-left: 10px;
                background-color: transparent;
            }
            QFormLayout {
                margin: 10px;
                spacing: 15px;
            }
            QFormLayout > QLabel {
                font-size: 13px;
                color: #E1E6EB;
                font-weight: normal;
                margin-right: 10px;
            }
            QFormLayout > QWidget {
                font-size: 13px;
                color: #E1E6EB;
            }
        """
        app.setStyleSheet(style_sheet)

    def set_light_theme(self):
        """Set a light theme that matches macOS style"""
        # Use Fusion style as base
        QApplication.setStyle(QStyleFactory.create("Fusion"))

        # Light color palette
        light_palette = QPalette()
        #  Set light color scheme.
        light_palette.setColor(QPalette.Window, Qt.white)
        light_palette.setColor(QPalette.WindowText, Qt.black)
        light_palette.setColor(QPalette.Base, QColor(240, 240, 240))
        light_palette.setColor(QPalette.AlternateBase, QColor(248, 248, 248))
        light_palette.setColor(QPalette.ToolTipBase, QColor(255, 255, 220))
        light_palette.setColor(QPalette.ToolTipText, Qt.black)
        light_palette.setColor(QPalette.Text, Qt.black)
        light_palette.setColor(QPalette.Button, QColor(230, 230, 230))
        light_palette.setColor(QPalette.ButtonText, Qt.black)
        light_palette.setColor(QPalette.BrightText, Qt.red)
        light_palette.setColor(QPalette.Link, QColor(0, 0, 255))
        light_palette.setColor(QPalette.Highlight, QColor(56, 134, 255))
        light_palette.setColor(QPalette.HighlightedText, Qt.white)
        light_palette.setColor(QPalette.Disabled, QPalette.WindowText, QColor(128, 128, 128))
        light_palette.setColor(QPalette.Disabled, QPalette.Text, QColor(128, 128, 128))
        light_palette.setColor(QPalette.Disabled, QPalette.ButtonText, QColor(128, 128, 128))
        QApplication.setPalette(light_palette)

        #  light stylesheet
        style_sheet = """
           QMainWindow, QDialog {
                background-color: #F0F0F0;
            }
            QLabel {
                font-size: 13px;
                color: #2D3748;
            }
            QLabel#title_label {
                font-size: 18px;
                font-weight: bold;
                color: #1A202C;
                padding: 10px;
            }
            QLabel#section_label {
                font-size: 15px;
                font-weight: bold;
                color: #1A202C;
                padding: 5px 0;
            }
            QLineEdit, QComboBox, QSpinBox {
                background-color: #FFFFFF;
                border: 1px solid #CBD5E0;
                border-radius: 4px;
                padding: 6px;
                selection-background-color: #4A5568;
                font-size: 13px;
                color: #2D3748;
            }
            QLineEdit:focus, QComboBox:focus, QSpinBox:focus {
                border: 1px solid #4A5568;
                box-shadow: 0 0 5px rgba(74, 85, 104, 0.5);
            }
            QComboBox::drop-down {
                subcontrol-origin: padding;
                subcontrol-position: top right;
                width: 20px;
                border-left: 1px solid #CBD5E0;
            }
            QComboBox QAbstractItemView {
                border: 1px solid #CBD5E0;
                background-color: #F7FAFC;
                selection-background-color: #4A5568;
                color: #2D3748;
                font-size: 13px;
            }
            QPushButton {
                background-color: #EDF2F7;
                color: #2D3748;
                border: 1px solid #CBD5E0;
                border-radius: 6px;
                padding: 10px 20px;
                font-size: 13px;
                font-weight: bold;
                transition: background-color 0.3s ease, color 0.3s ease, border-color 0.3s ease;
            }
            QPushButton:hover {
                background-color: #E2E8F0;
                color: #2D3748;
                border-color: #CBD5E0;
            }
            QPushButton:pressed {
                background-color: #CBD5E0;
                color: #2D3748;
                border-color: #CBD5E0;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.2) inset;
            }
            QPushButton:disabled {
                background-color: #F7FAFC;
                color: #A0AEC0;
                border-color: #CBD5E0;
            }
            QPushButton#primary_button {
                background-color: #4A7ACF;
                color: #FFFFFF;
                border-color: #4A7ACF;
            }
            QPushButton#primary_button:hover {
                background-color: #5A8ADF;
                color: #FFFFFF;
                border-color: #5A8ADF;
            }
            QPushButton#primary_button:pressed {
                background-color: #6A9AEF;
                color: #FFFFFF;
                border-color: #6A9AEF;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) inset;
            }
             QPushButton#danger_button {
                background-color: #CF4A4A;
                color: #FFFFFF;
                border-color: #CF4A4A;
                transition: background-color 0.3s ease;
            }
            QPushButton#danger_button:hover {
                background-color: #DF5A5A;
                color: #FFFFFF;
                border-color: #DF5A5A;
            }
            QPushButton#danger_button:pressed {
                background-color: #EF6A6A;
                color: #FFFFFF;
                border-color: #EF6A6A;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) inset;
            }
            QProgressBar {
                border: 1px solid #CBD5E0;
                border-radius: 4px;
                text-align: center;
                background-color: #F7FAFC;
                height: 24px;
            }
            QProgressBar::chunk {
                background-color: #4A7ACF;
                border-radius: 4px;
                transition: width 0.5s ease;
            }
            QTreeWidget, QTextEdit, QListWidget {
                background-color: #FFFFFF;
                alternate-background-color: #F7FAFC;
                border: 1px solid #CBD5E0;
                border-radius: 6px;
                selection-background-color: #4A5568;
                font-size: 13px;
                color: #2D3748;
            }
            QTreeWidget::item, QListWidget::item {
                padding: 8px;
            }
            QTreeWidget::item:selected, QListWidget::item:selected {
                background-color: #4A5568;
                color: #FFFFFF;
                border-radius: 4px;
            }
            QHeaderView::section {
                background-color: #F0F0F0;
                color: #1A202C;
                padding: 8px;
                border: none;
                border-right: 1px solid #CBD5E0;
                font-size: 13px;
                font-weight: bold;
            }
             QHeaderView::section:last {
                border-right: none;
            }
            QSplitter::handle {
                background-color: #CBD5E0;
                width: 1px;
            }
            QSplitter::handle:hover {
                background-color: #4A5568;
            }
            QTabWidget::pane {
                border: 1px solid #CBD5E0;
                border-radius: 6px;
                background-color: #F0F0F0;
            }
            QTabBar::tab {
                background-color: #EDF2F7;
                color: #2D3748;
                border-top-left-radius: 6px;
                border-top-right-radius: 6px;
                padding: 10px 20px;
                font-size: 13px;
                transition: background-color 0.3s ease, color 0.3s ease;
            }
            QTabBar::tab:selected {
                background-color: #FFFFFF;
                color: #1A202C;
            }
            QTabBar::tab:hover:!selected {
                background-color: #E2E8F0;
                color: #2D3748;
            }
            QToolBar {
                background-color: #F0F0F0;
                border: none;
                spacing: 10px;
                padding: 10px;
                border-radius: 6px;
            }
            QToolBar QToolButton {
                background-color: transparent;
                border-radius: 6px;
                padding: 8px;
                color: #2D3748;
                transition: background-color 0.3s ease, color 0.3s ease, border-color 0.3s ease;
            }
            QToolBar QToolButton:hover {
                background-color: #E2E8F0;
                color: #2D3748;
                border-color: #CBD5E0;
            }
            QToolBar QToolButton:pressed {background-color: #CBD5E0;
                color: #2D3748;
                border-color: #CBD5E0;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.2) inset;
            }
            QToolBar QToolButton:checked {
                background-color: #E2E8F0;
                color: #2D3748;
                border-color: #CBD5E0;
            }
            QStatusBar {
                background-color: #F0F0F0;
                color: #2D3748;
                padding: 8px;
                border-top: 1px solid #CBD5E0;
            }
            QStatusBar::item {
                border: none;
            }
            QMenu {
                background-color: #FFFFFF;
                border: 1px solid #CBD5E0;
                border-radius: 6px;
                padding: 5px;
                font-size: 13px;
                color: #2D3748;
            }
            QMenu::item {
                padding: 8px 25px 8px 30px;
                color: #2D3748;
                transition: background-color 0.3s ease, color 0.3s ease;
            }
            QMenu::item:selected {
                background-color: #E2E8F0;
                color: #2D3748;
                border-radius: 4px;
            }
            QMenu::separator {
                height: 1px;
                background-color: #CBD5E0;
                margin: 8px 0;
            }
            QSlider::groove:horizontal {
                height: 10px;
                background-color: #F7FAFC;
                border-radius: 5px;
                margin: 0;
            }
            QSlider::handle:horizontal {
                background-color: #4A7ACF;
                border: none;
                width: 20px;
                height: 20px;
                margin: -5px 0;
                border-radius: 10px;
                transition: transform 0.2s ease;
            }
            QSlider::handle:horizontal:hover {
                transform: scale(1.1);
            }
            QSlider::add-page:horizontal {
                background-color: #F7FAFC;
                border-radius: 5px;
            }
            QSlider::sub-page:horizontal {
                background-color: #E2E8F0;
                border-radius: 5px;
            }
            QCheckBox {
                color: #2D3748;
                font-size: 13px;
            }
            QCheckBox::indicator {
                width: 20px;
                height: 20px;
                background-color: #FFFFFF;
                border: 1px solid #CBD5E0;
                border-radius: 4px;
                transition: background-color 0.2s ease, border-color 0.2s ease;
            }
            QCheckBox::indicator:hover {
                border-color: #4A7ACF;
            }
            QCheckBox::indicator:checked {
                background-color: #4A7ACF;
                border-color: #4A7ACF;
                border-radius: 4px;
            }
            QCheckBox::indicator:checked:disabled {
                background-color: #A0AEC0;
                border-color: #A0AEC0;
            }
            QCheckBox::indicator:disabled {
                border-color: #CBD5E0;
            }
            QRadioButton {
                color: #2D3748;
                font-size: 13px;
            }
            QRadioButton::indicator {
                width: 20px;
                height: 20px;
                background-color: #FFFFFF;
                border: 1px solid #CBD5E0;
                border-radius: 10px;
                transition: background-color 0.2s ease, border-color 0.2s ease;
            }
            QRadioButton::indicator:hover {
                border-color: #4A7ACF;
            }
            QRadioButton::indicator:checked {
                background-color: #4A7ACF;
                border-color: #4A7ACF;
                border-radius: 10px;
            }
            QRadioButton::indicator:checked:disabled {
                background-color: #A0AEC0;
                border-color: #A0AEC0;
            }
            QRadioButton::indicator:disabled {
                border-color: #CBD5E0;
            }
            QGroupBox {
                border: 1px solid #CBD5E0;
                border-radius: 6px;
                margin-top: 24px;
                padding-top: 18px;
                font-size: 14px;
                font-weight: bold;
                color: #1A202C;
            }
            QGroupBox::title {
                subcontrol-origin: margin;
                subcontrol-position: top left;
                padding: 0 10px;
                margin-left: 10px;
                background-color: transparent;
            }
            QFormLayout {
                margin: 10px;
                spacing: 15px;
            }
            QFormLayout > QLabel {
                font-size: 13px;
                color: #2D3748;
                font-weight: normal;
                margin-right: 10px;
            }
            QFormLayout > QWidget {
                font-size: 13px;
                color: #2D3748;
            }
        """
        app.setStyleSheet(style_sheet)

    def toggle_theme(self):
        """Toggle between dark and light themes"""
        self.is_dark_theme = not self.is_dark_theme

        if self.is_dark_theme:
            self.set_dark_navy_theme()
        else:
            self.set_light_theme()

        # Save preference
        self.settings.setValue("dark_theme", self.is_dark_theme)

    def init_ui(self):
        # Create central widget and main layout
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QVBoxLayout(central_widget)
        main_layout.setSpacing(15)
        main_layout.setContentsMargins(15, 15, 15, 15)

        # Create toolbar
        self.create_toolbar()

        # Create a tab widget for the main interface
        self.tab_widget = QTabWidget()
        main_layout.addWidget(self.tab_widget)

        # Create tabs
        self.create_settings_tab()
        self.create_scan_results_tab()
        self.create_disk_space_tab()
        self.create_backup_tab()
        self.create_logs_tab()

        # Create status bar
        self.create_status_bar()

    def create_toolbar(self):
        """Create the application toolbar"""
        toolbar = QToolBar("Main Toolbar")
        toolbar.setMovable(False)
        toolbar.setIconSize(QSize(24, 24))
        self.addToolBar(toolbar)

        # Scan action
        scan_action = QAction("Scan", self)
        icon = self.style().standardIcon(QStyle.SP_FileDialogStart)
        scan_action.setIcon(icon)
        scan_action.triggered.connect(self.start_scan)
        toolbar.addAction(scan_action)

        # Move action
        move_action = QAction("Move", self)
        icon = self.style().standardIcon(QStyle.SP_ArrowRight)
        move_action.setIcon(icon)
        move_action.triggered.connect(self.move_selected_apps)
        toolbar.addAction(move_action)

        toolbar.addSeparator()

        # Backup action
        backup_action = QAction("Backup", self)
        icon = self.style().standardIcon(QStyle.SP_DialogSaveButton)
        backup_action.setIcon(icon)
        backup_action.triggered.connect(self.create_backup)
        toolbar.addAction(backup_action)

        # Restore action
        restore_action = QAction("Restore", self)
        icon = self.style().standardIcon(QStyle.SP_DialogOpenButton)
        restore_action.setIcon(icon)
        restore_action.triggered.connect(self.show_restore_dialog)
        toolbar.addAction(restore_action)

        toolbar.addSeparator()

        # Export action
        export_action = QAction("Export", self)
        icon = self.style().standardIcon(QStyle.SP_DialogSaveButton)
        export_action.setIcon(icon)
        export_action.triggered.connect(self.show_export_dialog)
        toolbar.addAction(export_action)

        toolbar.addSeparator()

        # Theme toggle action
        theme_action = QAction("Toggle Theme", self)
        icon = self.style().standardIcon(QStyle.SP_DialogHelpButton)
        theme_action.setIcon(icon)
        theme_action.triggered.connect(self.toggle_theme)
        toolbar.addAction(theme_action)

        # Settings action
        settings_action = QAction("Settings", self)
        icon = self.style().standardIcon(QStyle.SP_FileDialogDetailedView)
        settings_action.setIcon(icon)
        settings_action.triggered.connect(lambda: self.tab_widget.setCurrentIndex(0))
        toolbar.addAction(settings_action)

        # Spacer
        spacer = QWidget()
        spacer.setSizePolicy(1, 0)
        toolbar.addWidget(spacer)

        # Help action
        help_action = QAction("Help", self)
        icon = self.style().standardIcon(QStyle.SP_MessageBoxQuestion)
        help_action.setIcon(icon)
        help_action.triggered.connect(self.show_help)
        toolbar.addAction(help_action)

    def create_status_bar(self):
        """Create the status bar"""
        status_bar = QStatusBar()
        self.setStatusBar(status_bar)

        # Status label
        self.status_label = QLabel("Ready")
        status_bar.addWidget(self.status_label, 1)

        # Progress bar
        self.progress_bar = QProgressBar()
        self.progress_bar.setMaximumWidth(200)
        self.progress_bar.setValue(0)
        status_bar.addPermanentWidget(self.progress_bar)

    def create_settings_tab(self):
        """Create the settings tab"""
        settings_tab = QWidget()
        settings_layout = QVBoxLayout(settings_tab)
        settings_layout.setSpacing(15)

        # Source directories section
        source_group = QGroupBox("Source Directories")
        source_layout = QVBoxLayout(source_group)
        source_layout.setSpacing(10)

        # Add source directories
        self.source_dir_layouts = []
        for i, directory in enumerate(self.scan_dirs):
            dir_layout = QHBoxLayout()
            dir_textbox = QLineEdit(directory)
            dir_textbox.setObjectName(f"src_dir_{i}")
            dir_textbox.setMinimumWidth(400)

            dir_browse_btn = QPushButton("Browse")
            dir_browse_btn.clicked.connect(lambda checked, idx=i: self.browse_source_dir(idx))

            dir_layout.addWidget(dir_textbox)
            dir_layout.addWidget(dir_browse_btn)
            source_layout.addLayout(dir_layout)
            self.source_dir_layouts.append(dir_layout)

        # Add button to add more source directories
        add_source_btn = QPushButton("Add Source Directory")
        add_source_btn.clicked.connect(self.add_source_directory)
        source_layout.addWidget(add_source_btn)

        settings_layout.addWidget(source_group)

        # Target directories section
        target_group = QGroupBox("Target Directories")
        target_layout = QVBoxLayout(target_group)
        target_layout.setSpacing(10)

        # Setapp target
        setapp_layout = QHBoxLayout()
        setapp_label = QLabel("Setapp Destination:")
        setapp_label.setMinimumWidth(150)
        self.setapp_path = QLineEdit(self.target_dirs['setapp'])
        self.setapp_path.setMinimumWidth(400)

        setapp_browse = QPushButton("Browse")
        setapp_browse.clicked.connect(lambda: self.browse_target_dir('setapp'))

        setapp_layout.addWidget(setapp_label)
        setapp_layout.addWidget(self.setapp_path)
        setapp_layout.addWidget(setapp_browse)
        target_layout.addLayout(setapp_layout)

        # Applications target
        apps_layout = QHBoxLayout()
        apps_label = QLabel("Applications Destination:")
        apps_label.setMinimumWidth(150)
        self.apps_path = QLineEdit(self.target_dirs['applications'])
        self.apps_path.setMinimumWidth(400)

        apps_browse = QPushButton("Browse")
        apps_browse.clicked.connect(lambda: self.browse_target_dir('applications'))

        apps_layout.addWidget(apps_label)
        apps_layout.addWidget(self.apps_path)
        apps_layout.addWidget(apps_browse)
        target_layout.addLayout(apps_layout)

        # Add button for custom target categories
        add_category_btn = QPushButton("Add Custom Target Category")
        add_category_btn.clicked.connect(self.add_custom_target_category)
        target_layout.addWidget(add_category_btn)

        settings_layout.addWidget(target_group)

        # Scan options
        options_group = QGroupBox("Scan & Move Options")
        options_layout = QFormLayout(options_group)

        # Create backup option
        self.create_backup_checkbox = QCheckBox("Create backup before moving apps")
        self.create_backup_checkbox.setChecked(True)
        options_layout.addRow("", self.create_backup_checkbox)

        # Minimum app size filter
        self.min_size_filter = QComboBox()
        size_options = ["No minimum", "10 MB", "50 MB", "100 MB", "500 MB", "1 GB"]
        self.min_size_filter.addItems(size_options)
        options_layout.addRow("Minimum App Size:", self.min_size_filter)

        # App age filter
        self.app_age_filter = QComboBox()
        age_options = ["All apps", "Modified in last week", "Modified in last month", "Modified in last year"]
        self.app_age_filter.addItems(age_options)
        options_layout.addRow("App Age Filter:", self.app_age_filter)

        # Show symlinks option
        self.show_symlinks_checkbox = QCheckBox("Show symlinks in scan results")
        self.show_symlinks_checkbox.setChecked(True)
        options_layout.addRow("", self.show_symlinks_checkbox)

        # Minimize to tray option
        self.minimize_to_tray_checkbox = QCheckBox("Minimize to system tray when closing")
        self.minimize_to_tray_checkbox.setChecked(True)
        options_layout.addRow("", self.minimize_to_tray_checkbox)

        settings_layout.addWidget(options_group)

        # Action buttons
        action_layout = QHBoxLayout()

        spacer = QWidget()
        spacer.setSizePolicy(1, 0)
        action_layout.addWidget(spacer)

        # Save settings button
        save_settings_btn = QPushButton("Save Settings")
        save_settings_btn.setMinimumWidth(150)
        save_settings_btn.clicked.connect(self.save_settings)
        action_layout.addWidget(save_settings_btn)

        # Scan button
        scan_btn = QPushButton("Scan for Apps")
        scan_btn.setObjectName("primary_button")
        scan_btn.setMinimumWidth(150)
        scan_btn.setMinimumHeight(40)
        scan_btn.clicked.connect(self.start_scan)
        action_layout.addWidget(scan_btn)

        settings_layout.addLayout(action_layout)
        settings_layout.addStretch()

        # Add the tab
        self.tab_widget.addTab(settings_tab, "Settings")

    def create_scan_results_tab(self):
        """Create the scan results tab"""
        results_tab = QWidget()
        results_layout = QVBoxLayout(results_tab)
        results_layout.setSpacing(15)

        # Filter bar
        filter_layout = QHBoxLayout()

        # Search box
        search_label = QLabel("Search:")
        self.search_box = QLineEdit()
        self.search_box.setPlaceholderText("Filter by name...")
        self.search_box.textChanged.connect(self.apply_filters)

        # Category filter
        category_label = QLabel("Category:")
        self.category_filter = QComboBox()
        self.category_filter.addItems(["All", "Standard", "Setapp"])
        self.category_filter.currentTextChanged.connect(self.apply_filters)

        # Size filter
        size_label = QLabel("Size:")
        self.size_filter = QComboBox()
        self.size_filter.addItems(["All", "< 10 MB", "10-100 MB", "100-500 MB", "500 MB-1 GB", "> 1 GB"])
        self.size_filter.currentTextChanged.connect(self.apply_filters)

        # Status filter
        status_label = QLabel("Status:")
        self.status_filter = QComboBox()
        self.status_filter.addItems(["All", "Original", "Symlink"])
        self.status_filter.currentTextChanged.connect(self.apply_filters)

        filter_layout.addWidget(search_label)
        filter_layout.addWidget(self.search_box)
        filter_layout.addWidget(category_label)
        filter_layout.addWidget(self.category_filter)
        filter_layout.addWidget(size_label)
        filter_layout.addWidget(self.size_filter)
        filter_layout.addWidget(status_label)
        filter_layout.addWidget(self.status_filter)

        results_layout.addLayout(filter_layout)

        # Create a tree widget for apps with columns for details
        self.app_tree = QTreeWidget()
        self.app_tree.setAlternatingRowColors(True)
        self.app_tree.setSelectionMode(QTreeWidget.ExtendedSelection)
        self.app_tree.setSortingEnabled(True)
        self.app_tree.setRootIsDecorated(False)

        # Set up columns
        columns = ["Name", "Type", "Size", "Location", "Modified Date", "Status", "Version", "Last Used"]
        self.app_tree.setColumnCount(len(columns))
        self.app_tree.setHeaderLabels(columns)

        # Set column widths
        header = self.app_tree.header()
        header.setSectionResizeMode(0, QHeaderView.Stretch)  # Name column stretches
        for i in range(1, len(columns)):
            header.setSectionResizeMode(i, QHeaderView.ResizeToContents)

        # Context menu for the tree
        self.app_tree.setContextMenuPolicy(Qt.CustomContextMenu)
        self.app_tree.customContextMenuRequested.connect(self.show_context_menu)

        results_layout.addWidget(self.app_tree)

        # Disk space info
        disk_info_layout = QHBoxLayout()

        # Source disk info
        self.source_disk_label = QLabel("Source Disk: N/A")
        disk_info_layout.addWidget(self.source_disk_label)

        # Space to be freed
        self.space_freed_label = QLabel("Space to be freed: N/A")
        disk_info_layout.addWidget(self.space_freed_label)

        # Target disk info
        self.target_disk_label = QLabel("Target Disk: N/A")
        disk_info_layout.addWidget(self.target_disk_label)

        disk_info_layout.addStretch()

        # Selected apps size
        self.selected_size_label = QLabel("Selected: 0 apps (0 B)")
        disk_info_layout.addWidget(self.selected_size_label)

        results_layout.addLayout(disk_info_layout)

        # Action buttons
        buttons_layout = QHBoxLayout()

        # Selection buttons
        select_all_btn = QPushButton("Select All")
        select_all_btn.clicked.connect(self.select_all_apps)

        deselect_all_btn = QPushButton("Deselect All")
        deselect_all_btn.clicked.connect(self.deselect_all_apps)

        filter_buttons_layout = QHBoxLayout()
        filter_buttons_layout.addWidget(select_all_btn)
        filter_buttons_layout.addWidget(deselect_all_btn)

        # Batch move buttons
        move_large_btn = QPushButton("Move Large Apps")
        move_large_btn.clicked.connect(lambda: self.batch_move("large"))

        move_setapp_btn = QPushButton("Move All Setapp")
        move_setapp_btn.clicked.connect(lambda: self.batch_move("setapp"))

        batch_buttons_layout = QHBoxLayout()
        batch_buttons_layout.addWidget(move_large_btn)
        batch_buttons_layout.addWidget(move_setapp_btn)

        spacer = QWidget()
        spacer.setSizePolicy(1, 0)

        move_selected_btn = QPushButton("Move Selected Apps")
        move_selected_btn.setObjectName("primary_button")
        move_selected_btn.setMinimumWidth(180)
        move_selected_btn.setMinimumHeight(40)
        move_selected_btn.clicked.connect(self.move_selected_apps)

        buttons_layout.addLayout(filter_buttons_layout)
        buttons_layout.addLayout(batch_buttons_layout)
        buttons_layout.addWidget(spacer)
        buttons_layout.addWidget(move_selected_btn)

        results_layout.addLayout(buttons_layout)

        # Add the tab
        self.tab_widget.addTab(results_tab, "Scan Results")

    def create_disk_space_tab(self):
        """Create the disk space analysis tab"""
        disk_space_tab = QWidget()
        disk_layout = QVBoxLayout(disk_space_tab)
        disk_layout.setSpacing(15)

        # Source disk group
        source_group = QGroupBox("Source Disk")
        source_layout = QFormLayout(source_group)

        self.source_disk_path = QLabel("Path: N/A")
        self.source_disk_total = QLabel("Total Space: N/A")
        self.source_disk_used = QLabel("Used Space: N/A")
        self.source_disk_free = QLabel("Free Space: N/A")
        self.source_disk_percent = QLabel("Percent Used: N/A")

        source_layout.addRow("Disk Path:", self.source_disk_path)
        source_layout.addRow("Total Space:", self.source_disk_total)
        source_layout.addRow("Used Space:", self.source_disk_used)
        source_layout.addRow("Free Space:", self.source_disk_free)
        source_layout.addRow("Usage:", self.source_disk_percent)

        disk_layout.addWidget(source_group)

        # Target disk group
        target_group = QGroupBox("Target Disk")
        target_layout = QFormLayout(target_group)

        self.target_disk_path = QLabel("Path: N/A")
        self.target_disk_total = QLabel("Total Space: N/A")
        self.target_disk_used = QLabel("Used Space: N/A")
        self.target_disk_free = QLabel("Free Space: N/A")
        self.target_disk_percent = QLabel("Percent Used: N/A")

        target_layout.addRow("Disk Path:", self.target_disk_path)
        target_layout.addRow("Total Space:", self.target_disk_total)
        target_layout.addRow("Used Space:", self.target_disk_used)
        target_layout.addRow("Free Space:", self.target_disk_free)
        target_layout.addRow("Usage:", self.target_disk_percent)

        disk_layout.addWidget(target_group)

        # After move impact group
        impact_group = QGroupBox("Move Impact")
        impact_layout = QFormLayout(impact_group)

        self.total_app_size = QLabel("Total App Size to Move: N/A")
        self.source_space_impact = QLabel("Impact on Source Disk: N/A")
        self.target_space_impact = QLabel("Impact on Target Disk: N/A")

        impact_layout.addRow("Total Size:", self.total_app_size)
        impact_layout.addRow("Source Impact:", self.source_space_impact)
        impact_layout.addRow("Target Impact:", self.target_space_impact)

        disk_layout.addWidget(impact_group)
        disk_layout.addStretch()

        # Add the tab
        self.tab_widget.addTab(disk_space_tab, "Disk Space")

    def create_backup_tab(self):
        """Create the backup management tab"""
        backup_tab = QWidget()
        backup_layout = QVBoxLayout(backup_tab)
        backup_layout.setSpacing(15)

        # Backup list
        backup_list_group = QGroupBox("Available Backups")
        backup_list_layout = QVBoxLayout(backup_list_group)

        self.backup_tree = QTreeWidget()
        self.backup_tree.setAlternat

SyntaxError: cannot assign to function call (<ipython-input-4-8aa5b2e63ecc>, line 239)

In [ ]:
import os
import sys
import shutil
import time
import datetime
import json
import subprocess
import tempfile
import platform
import re
from concurrent.futures import ThreadPoolExecutor

# macOS Integration for Icons and Last Used Date
import objc

# PyQt5 Imports
from PyQt5.QtWidgets import (
    QApplication,
    QMainWindow,
    QPushButton,
    QVBoxLayout,
    QHBoxLayout,
    QWidget,
    QLabel,
    QCheckBox,
    QFileDialog,
    QTreeWidget,
    QTreeWidgetItem,
    QMessageBox,
    QProgressBar,  # For progress bars
    QLineEdit,
    QHeaderView,
    QSplitter,
    QFrame,
    QTabWidget,
    QStyle,
    QStyleFactory,
    QMenu,
    QAction,
    QSystemTrayIcon,
    QDialog,
    QRadioButton,
    QComboBox,
    QSlider,
    QToolBar,
    QStatusBar,
    QTextEdit,
    QGroupBox,
    QFormLayout,
    QSpinBox,
)
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QSize, QTimer, QSettings, QDateTime, QSortFilterProxyModel
# For animations (if needed)
# from PyQt5.QtCore import QPropertyAnimation
from PyQt5.QtGui import QPalette, QColor, QFont, QIcon, QPixmap, QImage, QPainter, QBrush, QFontMetrics

class AppInfo:
    """
    Represents information about an application.
    """
    def __init__(self, path):
        self.path = path
        self.name = os.path.basename(path)
        self.location = os.path.dirname(path)
        self.size = self.get_size()
        self.modified = self.get_modified_time()
        self.is_symlink = os.path.islink(path)
        self.target = os.readlink(path) if self.is_symlink else ""
        self.app_type = "Setapp" if "/Setapp/" in path else "Standard"
        self.version = self.get_version()
        self.last_used = self.get_last_used()
        self.icon = self.get_icon()
        self.id = self.get_bundle_id()

    # ... (other methods)

    def get_icon(self):
        """Get app icon using NSWorkspace (macOS specific)"""
        try:
            NSWorkspace = objc.lookUpClass('NSWorkspace')
            sharedWorkspace = NSWorkspace.sharedWorkspace()
            url = objc.lookUpClass('NSURL').fileURLWithPath_(self.path)
            icon = sharedWorkspace.iconForFile_(url.path())

            img = icon.TIFFRepresentation()
            qimage = QImage.fromData(img)
            qpixmap = QPixmap.fromImage(qimage)
            return QIcon(qpixmap)

        except Exception:
            return None  # Or a default icon

    def get_last_used(self):
        """Get app last used date using NSWorkspace (more reliable)"""
        try:
            NSWorkspace = objc.lookUpClass('NSWorkspace')
            sharedWorkspace = NSWorkspace.sharedWorkspace()
            url = objc.lookUpClass('NSURL').fileURLWithPath_(self.path)
            metadata = sharedWorkspace.metadataForURL_error_(url, None)[0]

            last_used_date = metadata.objectForKey_('NSMetadataItemLastUsedDate')
            if last_used_date:
                return last_used_date.description()
            return "Unknown"

        except Exception:
            return "Unknown"

    # ... (rest of the AppInfo class)

# ... (BackupManager, LogManager, DiskSpaceAnalyzer classes)

class ScanThread(QThread):
    # ... (signals)

    def __init__(self, scan_dirs, progress_bar=None):  # Add progress_bar
        super().__init__()
        self.scan_dirs = scan_dirs
        self.stopped = False
        self.executor = ThreadPoolExecutor(max_workers=4)
        self.progress_bar = progress_bar  # Store progress bar